# Task 10 — HR Chatbot Memory and Permissions

**Project:** CortexHR / Employee ML Project  
**Notebook:** `09_chatbot_memory_permissions.ipynb`

## Objective
Validate the Task 10 conversational HR chatbot using the supplied dialogue dataset, authenticated demo users, session memory, role-based permissions and mock HR service.

This notebook tests the implemented FastAPI application. It does not replace code in `05_Starter/api_skeleton/app/`.

## Scope
- Session-based memory and entity/slot reuse.
- Multi-turn follow-up intent handling.
- Authentication through the `X-Demo-User` header.
- Role resolution from trusted `demo_users.csv` data.
- Intent authorization using `chatbot_responses.csv`.
- Employee privacy and salary-disclosure refusal.
- Supplied dialogue acceptance tests DLG-01 to DLG-12.
- Security and mock-service route checks.

## Before running
Start both services in separate terminals and keep them running.

**Terminal 1 — Mock HR service**
```powershell
cd 05_Starter\mock_services
uvicorn mock_server:app --port 9000
```

**Terminal 2 — CortexHR API**
```powershell
cd 05_Starter\api_skeleton
uvicorn app.main:app --reload --port 8001
```

Run this notebook from the repository root or `notebooks/`. The next cell locates the repository automatically.

## 1. Imports and project paths
Locate the project using the supplied dialogue CSV and configure paths for data, reports, mock data and the API package.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import httpx
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "data" / "chatbot_dialogues.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Open this notebook from inside employee-ml-project."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
API_DIR = PROJECT_ROOT / "05_Starter" / "api_skeleton"
MOCK_DIR = PROJECT_ROOT / "05_Starter" / "mock_services"
MOCK_DATA_DIR = MOCK_DIR / "mock_data"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
if str(API_DIR) not in sys.path:
    sys.path.insert(0, str(API_DIR))

API_BASE_URL = "http://127.0.0.1:8001"
MOCK_BASE_URL = "http://127.0.0.1:9000"

print("Project root:", PROJECT_ROOT)
print("API:", API_BASE_URL)
print("Mock HR service:", MOCK_BASE_URL)

Project root: C:\Users\kalpe\Documents\employee-ml-project
API: http://127.0.0.1:8001
Mock HR service: http://127.0.0.1:9000


## 2. Verify supplied Task 10 files
This notebook reads the project's existing CSV files. It does not generate replacement HR or dialogue data.

In [2]:
required_files = [
    DATA_DIR / "chatbot_dialogues.csv",
    DATA_DIR / "chatbot_responses.csv",
    DATA_DIR / "chatbot_nlu_training.csv",
    DATA_DIR / "chatbot_test_set.csv",
    DATA_DIR / "hr_faq.csv",
    MOCK_DATA_DIR / "demo_users.csv",
    MOCK_DATA_DIR / "employees.csv",
    MOCK_DATA_DIR / "payslips.csv",
    MOCK_DATA_DIR / "attendance_monthly.csv",
    MOCK_DATA_DIR / "leave_balances.csv",
    MOCK_DATA_DIR / "leave_requests.csv",
    MOCK_DATA_DIR / "holidays.csv",
]
file_status = pd.DataFrame([
    {"file": str(p.relative_to(PROJECT_ROOT)), "exists": p.exists()}
    for p in required_files
])
display(file_status)
missing = file_status.loc[~file_status["exists"], "file"].tolist()
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))

,file,exists
0,data\chatbot_dialogues.csv,True
1,data\chatbot_responses.csv,True
2,data\chatbot_nlu_training.csv,True
3,data\chatbot_test_set.csv,True
4,data\hr_faq.csv,True
5,05_Starter\mock_services\mock_data\demo_users.csv,True
6,05_Starter\mock_services\mock_data\employees.csv,True
7,05_Starter\mock_services\mock_data\payslips.csv,True
8,05_Starter\mock_services\mock_data\attendance_...,True
9,05_Starter\mock_services\mock_data\leave_balan...,True


## 3. Load source datasets
The dialogue CSV is the acceptance-test source. Response rules define login, allowed roles, service calls and configured failure messages.

In [3]:
dialogues = pd.read_csv(DATA_DIR / "chatbot_dialogues.csv")
response_rules = pd.read_csv(DATA_DIR / "chatbot_responses.csv")
demo_users = pd.read_csv(MOCK_DATA_DIR / "demo_users.csv")
employees = pd.read_csv(MOCK_DATA_DIR / "employees.csv")

print("Dialogue columns:", dialogues.columns.tolist())
print("Response-rule columns:", response_rules.columns.tolist())
print("Dialogue rows:", len(dialogues))
print("Dialogue count:", dialogues["dialogue_id"].nunique())
display(dialogues)

Dialogue columns: ['dialogue_id', 'turn_no', 'speaker', 'user_role', 'text', 'expected_intent', 'context_note']
Response-rule columns: ['intent', 'response_template', 'requires_login', 'allowed_roles', 'backend_or_service_call', 'on_failure', 'policy_note']
Dialogue rows: 28
Dialogue count: 12


,dialogue_id,turn_no,speaker,user_role,text,expected_intent,context_note
0,DLG-01,1,USER,employee,how many casual leaves do I have left,leave_balance,NaN
1,DLG-01,2,USER,employee,and sick leave?,leave_balance,follow-up: reuse entity/slot from previous turn
2,DLG-01,3,USER,employee,thanks,thanks,NaN
3,DLG-02,1,USER,employee,is my March payslip ready,payslip_status,NaN
4,DLG-02,2,USER,employee,who is my manager,my_manager,NaN
5,DLG-03,1,USER,employee,how do I apply for leave,apply_leave_howto,NaN
6,DLG-03,2,USER,employee,what is the leave policy,hr_policy_leave,NaN
7,DLG-03,3,USER,employee,ok bye,goodbye,NaN
8,DLG-04,1,USER,employee,show my attendance for last month,attendance_summary,NaN
9,DLG-04,2,USER,employee,any unusual days?,unusual_attendance,NaN


## 4. Inspect dialogue acceptance criteria
Each dialogue must run sequentially with one session ID per dialogue. This preserves context between turns while isolating separate conversations.

In [4]:
dialogue_overview = (
    dialogues.groupby(["dialogue_id", "user_role"], sort=False)
    .agg(turns=("turn_no", "count"))
    .reset_index()
)
display(dialogue_overview)
print("Total turns:", len(dialogues))

,dialogue_id,user_role,turns
0,DLG-01,employee,3
1,DLG-02,employee,2
2,DLG-03,employee,3
3,DLG-04,employee,2
4,DLG-05,employee,2
5,DLG-06,employee,3
6,DLG-07,hr,3
7,DLG-08,hr,2
8,DLG-09,hr,2
9,DLG-10,hr,2


Total turns: 28


## 5. Verify trusted demo identities
The test suite uses real identities from `demo_users.csv`: EMP-0002 for employee scenarios and EMP-0011 for HR scenarios. The request body does not supply a trusted role.

In [5]:
demo_users["role_normalized"] = demo_users["role"].astype(str).str.strip().str.upper()

def verify_demo_identity(employee_id, expected_role):
    rows = demo_users[demo_users["employee_id"].astype(str) == employee_id]
    if rows.empty:
        raise ValueError(f"Demo identity {employee_id} not found.")
    actual_role = str(rows.iloc[0]["role"]).strip().upper()
    if actual_role != expected_role:
        raise ValueError(f"{employee_id}: expected {expected_role}, found {actual_role}")

EMPLOYEE_TEST_ID = "EMP-0002"
HR_TEST_ID = "EMP-0011"
verify_demo_identity(EMPLOYEE_TEST_ID, "EMPLOYEE")
verify_demo_identity(HR_TEST_ID, "HR")

print("Employee identity:", EMPLOYEE_TEST_ID)
print("HR identity:", HR_TEST_ID)
print("Available HR accounts:", int((demo_users["role_normalized"] == "HR").sum()))
print("Available ADMIN accounts:", int((demo_users["role_normalized"] == "ADMIN").sum()))

Employee identity: EMP-0002
HR identity: EMP-0011
Available HR accounts: 19
Available ADMIN accounts: 3


## 6. Check service connectivity
If either service is unavailable, start it using the commands in the introduction and rerun this cell.

In [6]:
def check_service(url, label):
    try:
        response = httpx.get(url, timeout=5.0)
        print(f"{label}: reachable (HTTP {response.status_code})")
        return response.status_code
    except httpx.HTTPError as exc:
        raise RuntimeError(
            f"{label} is not reachable at {url}. Start the service and rerun."
        ) from exc

api_docs_status = check_service(f"{API_BASE_URL}/docs", "CortexHR API")
mock_docs_status = check_service(f"{MOCK_BASE_URL}/docs", "Mock HR service")

CortexHR API: reachable (HTTP 200)
Mock HR service: reachable (HTTP 200)


## 7. Reusable chatbot API client
Authenticated requests use `X-Demo-User`. Dialogue requests send only `message` and `session_id`; they do not send a body role.

In [7]:
def chat_request(message, session_id, employee_id=EMPLOYEE_TEST_ID, extra_body=None):
    headers = {}
    if employee_id is not None:
        headers["X-Demo-User"] = employee_id
    body = {"message": message, "session_id": session_id}
    if extra_body:
        body.update(extra_body)
    return httpx.post(
        f"{API_BASE_URL}/v1/chat",
        headers=headers,
        json=body,
        timeout=20.0,
    )

## 8. Execute all supplied dialogues (DLG-01 to DLG-12)
Each dialogue gets a unique session. Employee dialogues use EMP-0002 and HR dialogues use EMP-0011. Actual intents are compared with the supplied `expected_intent` values.

In [8]:
dialogue_results = []

for dialogue_id, group in dialogues.groupby("dialogue_id", sort=False):
    group = group.sort_values("turn_no")
    session_id = f"task10-{dialogue_id}"
    role = str(group.iloc[0]["user_role"]).strip().lower()
    employee_id = HR_TEST_ID if role == "hr" else EMPLOYEE_TEST_ID

    for _, row in group.iterrows():
        message = str(row["text"])
        expected = str(row["expected_intent"])
        try:
            response = chat_request(message, session_id, employee_id)
            response.raise_for_status()
            payload = response.json()
            actual = payload.get("intent", "MISSING_INTENT")
            reply = payload.get("reply", "")
            http_status = response.status_code
            status = "PASS" if actual == expected else "FAIL"
        except Exception as exc:
            actual, reply, http_status, status = "ERROR", repr(exc), None, "ERROR"

        dialogue_results.append({
            "dialogue_id": dialogue_id,
            "turn_no": row["turn_no"],
            "user_role": role,
            "employee_id": employee_id,
            "message": message,
            "expected_intent": expected,
            "actual_intent": actual,
            "http_status": http_status,
            "status": status,
            "reply": reply,
        })
        print(
            f"{dialogue_id} | Turn {row['turn_no']} | {status} | "
            f"Expected: {expected} | Actual: {actual}"
        )

dialogue_results_df = pd.DataFrame(dialogue_results)
dialogue_results_path = REPORTS_DIR / "task10_dialogue_test_results.csv"
dialogue_results_df.to_csv(dialogue_results_path, index=False)

total_turns = len(dialogue_results_df)
passed_turns = int((dialogue_results_df["status"] == "PASS").sum())
failed_turns = int((dialogue_results_df["status"] == "FAIL").sum())
error_turns = int((dialogue_results_df["status"] == "ERROR").sum())
accuracy = 100 * passed_turns / total_turns if total_turns else 0.0

dialogue_summary = pd.DataFrame([{
    "total_turns": total_turns,
    "passed": passed_turns,
    "failed": failed_turns,
    "errors": error_turns,
    "accuracy_percent": round(accuracy, 2),
}])
display(dialogue_summary)
print("Detailed results:", dialogue_results_path)

failed_rows = dialogue_results_df[dialogue_results_df["status"] != "PASS"]
if not failed_rows.empty:
    display(failed_rows[[
        "dialogue_id", "turn_no", "message",
        "expected_intent", "actual_intent", "status"
    ]])

DLG-01 | Turn 1 | PASS | Expected: leave_balance | Actual: leave_balance
DLG-01 | Turn 2 | PASS | Expected: leave_balance | Actual: leave_balance
DLG-01 | Turn 3 | PASS | Expected: thanks | Actual: thanks
DLG-02 | Turn 1 | PASS | Expected: payslip_status | Actual: payslip_status
DLG-02 | Turn 2 | PASS | Expected: my_manager | Actual: my_manager
DLG-03 | Turn 1 | PASS | Expected: apply_leave_howto | Actual: apply_leave_howto
DLG-03 | Turn 2 | PASS | Expected: hr_policy_leave | Actual: hr_policy_leave
DLG-03 | Turn 3 | PASS | Expected: goodbye | Actual: goodbye
DLG-04 | Turn 1 | PASS | Expected: attendance_summary | Actual: attendance_summary
DLG-04 | Turn 2 | PASS | Expected: unusual_attendance | Actual: unusual_attendance
DLG-05 | Turn 1 | PASS | Expected: salary_of_other | Actual: salary_of_other
DLG-05 | Turn 2 | PASS | Expected: payslip_status | Actual: payslip_status
DLG-06 | Turn 1 | PASS | Expected: greet | Actual: greet
DLG-06 | Turn 2 | PASS | Expected: chitchat | Actual: chitc

,total_turns,passed,failed,errors,accuracy_percent
0,28,28,0,0,100.0


Detailed results: C:\Users\kalpe\Documents\employee-ml-project\reports\task10_dialogue_test_results.csv


## 9. Assert dialogue acceptance criteria
The supplied acceptance suite contains 28 turns. A successful run must have 28 matching intents and zero errors.

In [9]:
assert total_turns == len(dialogues), (
    f"Expected {len(dialogues)} turns, ran {total_turns}."
)
assert failed_turns == 0, (
    f"{failed_turns} turn(s) failed intent comparison. Review the failure table."
)
assert error_turns == 0, (
    f"{error_turns} request(s) errored. Check both services and API logs."
)
print(f"PASS: {passed_turns}/{total_turns} dialogue turns matched.")
print(f"Intent accuracy: {accuracy:.2f}%")

PASS: 28/28 dialogue turns matched.
Intent accuracy: 100.00%


## 10. Security test — another employee's salary
An authenticated employee must not be able to obtain another employee's salary or private payroll information.

In [10]:
salary_response = chat_request(
    "What is Priya Nair salary?",
    "task10-security-salary-privacy",
    EMPLOYEE_TEST_ID,
)
salary_response.raise_for_status()
salary_payload = salary_response.json()

salary_privacy_pass = (
    salary_payload.get("intent") == "salary_of_other"
    and "not permitted" in str(salary_payload.get("reply", "")).lower()
)
display(pd.DataFrame([{
    "test": "Other employee salary refusal",
    "intent": salary_payload.get("intent"),
    "reply": salary_payload.get("reply"),
    "passed": salary_privacy_pass,
}]))
assert salary_privacy_pass, "Salary privacy test failed."

,test,intent,reply,passed
0,Other employee salary refusal,salary_of_other,This request is not permitted.,True


## 11. Security test — request-body role spoofing
EMP-0002 is an EMPLOYEE. Sending `user_role='HR'` in the JSON body must not grant HR-only access.

In [11]:
spoof_response = chat_request(
    "Suggest a salary band for Data Analyst",
    "task10-security-role-spoof",
    EMPLOYEE_TEST_ID,
    extra_body={"user_role": "HR"},
)
spoof_response.raise_for_status()
spoof_payload = spoof_response.json()

role_spoof_pass = (
    spoof_payload.get("intent") == "salary_band_query"
    and "permission" in str(spoof_payload.get("reply", "")).lower()
)
display(pd.DataFrame([{
    "test": "Request-body role spoofing denied",
    "intent": spoof_payload.get("intent"),
    "reply": spoof_payload.get("reply"),
    "passed": role_spoof_pass,
}]))
assert role_spoof_pass, "Role-spoofing test failed."

,test,intent,reply,passed
0,Request-body role spoofing denied,salary_band_query,You do not have permission to access this info...,True


## 12. Security test — authentication required
A leave-balance request without the `X-Demo-User` header must be denied.

In [12]:
unauth_response = chat_request(
    "How many casual leaves do I have left?",
    "task10-security-no-login",
    employee_id=None,
)
unauth_response.raise_for_status()
unauth_payload = unauth_response.json()

authentication_pass = (
    unauth_payload.get("intent") == "leave_balance"
    and "authenticate" in str(unauth_payload.get("reply", "")).lower()
)
display(pd.DataFrame([{
    "test": "Authentication required",
    "intent": unauth_payload.get("intent"),
    "reply": unauth_payload.get("reply"),
    "passed": authentication_pass,
}]))
assert authentication_pass, "Authentication test failed."

,test,intent,reply,passed
0,Authentication required,leave_balance,Please authenticate before accessing this info...,True


## 13. Security test — genuine HR authorization
A real HR demo account should pass authorization for the HR-only salary-band intent. The supplied mock server does not expose a salary-band analytics route, so the expected response is the configured service failure message, not a permission denial or fabricated salary band.

In [13]:
hr_response = chat_request(
    "Suggest a salary band for Data Analyst",
    "task10-security-genuine-hr",
    HR_TEST_ID,
)
hr_response.raise_for_status()
hr_payload = hr_response.json()

genuine_hr_pass = (
    hr_payload.get("intent") == "salary_band_query"
    and "permission" not in str(hr_payload.get("reply", "")).lower()
)
display(pd.DataFrame([{
    "test": "Genuine HR authorization",
    "intent": hr_payload.get("intent"),
    "reply": hr_payload.get("reply"),
    "passed": genuine_hr_pass,
}]))
assert genuine_hr_pass, "Genuine HR authorization test failed."

,test,intent,reply,passed
0,Genuine HR authorization,salary_band_query,"Sorry, I could not fetch that right now. Pleas...",True


## 14. Slot extraction and session-memory checks
Import the actual memory service and verify extraction of month, employee name, department and job title. Then check that the session retains authenticated identity, the latest department and the previous intent.

In [14]:
from app.memory_service import (
    extract_slots,
    update_session_memory,
    get_session_slots,
    clear_session_memory,
)

slot_checks = {
    "month_department": extract_slots("Show attendance for March 2026 in Sales"),
    "job_title": extract_slots("Suggest a salary band for Data Analyst"),
    "employee_name": extract_slots("What is Jatin Jayaraman salary?"),
}
display(pd.DataFrame([
    {"case": key, "slots": str(value)}
    for key, value in slot_checks.items()
]))

assert slot_checks["month_department"].get("month") == "March 2026"
assert slot_checks["month_department"].get("department") == "Sales"
assert slot_checks["job_title"].get("job_title") == "Data Analyst"
assert slot_checks["employee_name"].get("employee") == "Jatin Jayaraman"

test_session = "task10-notebook-memory-check"
clear_session_memory(test_session, EMPLOYEE_TEST_ID)
update_session_memory(
    test_session, EMPLOYEE_TEST_ID,
    "Who is at risk of leaving in Sales?",
    intent="attrition_risk_query",
)
update_session_memory(
    test_session, EMPLOYEE_TEST_ID,
    "And in Engineering?",
    intent="attrition_risk_query",
)
remembered = get_session_slots(test_session, EMPLOYEE_TEST_ID)
display(pd.DataFrame([remembered]))

assert remembered.get("authenticated_employee_id") == EMPLOYEE_TEST_ID
assert remembered.get("department") == "Engineering"
assert remembered.get("previous_intent") == "attrition_risk_query"
clear_session_memory(test_session, EMPLOYEE_TEST_ID)
print("PASS: slot extraction and memory assertions.")

,case,slots
0,month_department,"{'month': 'March 2026', 'department': 'Sales'}"
1,job_title,{'job_title': 'Data Analyst'}
2,employee_name,{'employee': 'Jatin Jayaraman'}


,authenticated_employee_id,department,previous_intent
0,EMP-0002,Engineering,attrition_risk_query


PASS: slot extraction and memory assertions.


## 15. Direct mock HR service route checks
The supplied mock server exposes the seven routes listed below. These checks call the mock service directly with the authenticated employee header. A route may return a non-200 response if its source data does not contain the requested month or record; inspect the response detail rather than assuming data exists.

In [15]:
mock_headers = {"X-Demo-User": EMPLOYEE_TEST_ID}
mock_route_cases = [
    ("Employee profile", "/api/v1/employees/me", None),
    ("Team members", "/api/v1/employees/me/team", None),
    ("Leave balance", "/api/v1/leave/balance", {"employee": "me"}),
    ("Leave requests", "/api/v1/leave/requests", {"employee": "me", "limit": 5}),
    ("April 2026 payslip", "/api/v1/payslips/me/2026-04", None),
    ("September 2026 attendance", "/api/v1/attendance/summary",
     {"employee": "me", "month": "2026-09"}),
    ("Holiday list", "/api/v1/holidays", None),
]

mock_route_results = []
for label, path, params in mock_route_cases:
    try:
        response = httpx.get(
            f"{MOCK_BASE_URL}{path}",
            headers=mock_headers,
            params=params,
            timeout=10.0,
        )
        mock_route_results.append({
            "route": label,
            "http_status": response.status_code,
            "passed": response.status_code == 200,
            "detail": "OK" if response.status_code == 200 else response.text[:300],
        })
    except httpx.HTTPError as exc:
        mock_route_results.append({
            "route": label,
            "http_status": None,
            "passed": False,
            "detail": repr(exc),
        })

mock_route_results_df = pd.DataFrame(mock_route_results)
display(mock_route_results_df)

,route,http_status,passed,detail
0,Employee profile,200,True,OK
1,Team members,200,True,OK
2,Leave balance,200,True,OK
3,Leave requests,200,True,OK
4,April 2026 payslip,200,True,OK
5,September 2026 attendance,200,True,OK
6,Holiday list,200,True,OK


## 16. Save evidence CSV files
Save the detailed dialogue results, compact summary, security outcomes and direct mock-route results under the project's `reports/` directory.

In [16]:
security_results = pd.DataFrame([
    {"test": "Other employee salary refusal", "passed": salary_privacy_pass,
     "intent": salary_payload.get("intent"), "reply": salary_payload.get("reply")},
    {"test": "Request-body role spoofing denied", "passed": role_spoof_pass,
     "intent": spoof_payload.get("intent"), "reply": spoof_payload.get("reply")},
    {"test": "Authentication required", "passed": authentication_pass,
     "intent": unauth_payload.get("intent"), "reply": unauth_payload.get("reply")},
    {"test": "Genuine HR authorization", "passed": genuine_hr_pass,
     "intent": hr_payload.get("intent"), "reply": hr_payload.get("reply")},
])

summary_path = REPORTS_DIR / "task10_dialogue_test_summary.csv"
security_path = REPORTS_DIR / "task10_security_test_results.csv"
routes_path = REPORTS_DIR / "task10_mock_service_route_checks.csv"

dialogue_summary.to_csv(summary_path, index=False)
security_results.to_csv(security_path, index=False)
mock_route_results_df.to_csv(routes_path, index=False)

print("Evidence saved:")
for path in [dialogue_results_path, summary_path, security_path, routes_path]:
    print("-", path.relative_to(PROJECT_ROOT))
display(security_results)

Evidence saved:
- reports\task10_dialogue_test_results.csv
- reports\task10_dialogue_test_summary.csv
- reports\task10_security_test_results.csv
- reports\task10_mock_service_route_checks.csv


,test,passed,intent,reply
0,Other employee salary refusal,True,salary_of_other,This request is not permitted.
1,Request-body role spoofing denied,True,salary_band_query,You do not have permission to access this info...
2,Authentication required,True,leave_balance,Please authenticate before accessing this info...
3,Genuine HR authorization,True,salary_band_query,"Sorry, I could not fetch that right now. Pleas..."


## 17. Final validation summary
The dialogue acceptance suite, security checks, slot/memory assertions and mock-route checks are reported separately. Analytics intents for which the supplied mock server has no backend must return the configured failure message rather than invented predictions or recommendations.

In [17]:
final_summary = {
    "dialogue_total": total_turns,
    "dialogue_passed": passed_turns,
    "dialogue_failed": failed_turns,
    "dialogue_errors": error_turns,
    "dialogue_accuracy_percent": round(accuracy, 2),
    "security_tests_passed": int(security_results["passed"].sum()),
    "security_tests_total": len(security_results),
    "slot_memory_checks": "PASS",
    "mock_routes_http_200": int(mock_route_results_df["passed"].sum()),
    "mock_routes_checked": len(mock_route_results_df),
}
display(pd.DataFrame([final_summary]))

assert total_turns == 28
assert passed_turns == 28
assert failed_turns == 0
assert error_turns == 0
assert security_results["passed"].all()
print("TASK 10 VALIDATION: ASSERTED DIALOGUE, SECURITY AND MEMORY CHECKS PASSED.")

,dialogue_total,dialogue_passed,dialogue_failed,dialogue_errors,dialogue_accuracy_percent,security_tests_passed,security_tests_total,slot_memory_checks,mock_routes_http_200,mock_routes_checked
0,28,28,0,0,100.0,4,4,PASS,7,7


TASK 10 VALIDATION: ASSERTED DIALOGUE, SECURITY AND MEMORY CHECKS PASSED.
